## 02b_scd_dimensions — RCM Intelligence Platform
### Purpose
Builds and maintains all four SCD dimensions in the Silver layer.
These dimensions are the backbone of the Gold star schema.

### Dimensions built
| Dimension    | SCD Type | Tracks |
|---|---|---|
| dim_drg      | Type 1   | DRG descriptions, weight updates |
| dim_hospital | Type 2   | Ownership changes, type reclassification |
| dim_payer    | Type 2   | Contract rate changes, policy updates |
| dim_provider | Type 2   | Specialty, location, group practice changes |
| dim_date     | Static   | Calendar table — fiscal periods, CMS quarters |

### Inputs
- rcm_platform.rcm_silver.inpatient_claims_enriched
- rcm_platform.rcm_bronze.hospital_info_raw

### Outputs
- rcm_platform.rcm_silver.dim_drg
- rcm_platform.rcm_silver.dim_hospital
- rcm_platform.rcm_silver.dim_payer
- rcm_platform.rcm_silver.dim_provider
- rcm_platform.rcm_silver.dim_date

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Silver |
| Run order  | 5 — after 02a_claims_transform |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA + SCD HELPER FUNCTIONS
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, sha2, concat_ws, current_date,
    when, coalesce, expr, trim, date_sub
)
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "02b_scd_dimensions"

# SCD constants from config
END_OF_TIME    = SCD_END_OF_TIME    # "9999-12-31"
INITIAL_DATE   = SCD_INITIAL_DATE   # "2024-01-01"

print(f"Batch ID     : {BATCH_ID}")
print(f"Batch date   : {BATCH_DATE}")
print(f"End of time  : {END_OF_TIME}")
print(f"Initial date : {INITIAL_DATE}")


def compute_row_hash(*col_names) -> col:
    """
    Computes a SHA2 hash of all tracked columns.
    Used to detect changes between runs for SCD Type 2.
    If the hash changes — the row has changed — expire and insert.
    """
    return sha2(
        concat_ws("|", *[coalesce(col(c).cast(StringType()), lit("")) 
                         for c in col_names]),
        256
    )


def make_surrogate_key(*col_names) -> col:
    """
    Creates a surrogate key from natural key + effective date.
    Guarantees uniqueness even if the same provider appears
    multiple times across different time periods.
    """
    return sha2(
        concat_ws("|", *[coalesce(col(c).cast(StringType()), lit(""))
                         for c in col_names]),
        256
    )


print("SCD helper functions loaded")

In [0]:
# ============================================================
# DIM_DRG — SCD TYPE 1
# Reference dimension — always reflects current CMS definition
# On change: overwrite in place (no history kept)
# ============================================================

print("=" * 55)
print("  BUILDING dim_drg — SCD TYPE 1")
print("=" * 55)

df_drg_source = spark.table(TBL_SILVER_CLAIMS).select(
    col("drg_code"),
    col("drg_description"),
    col("drg_severity_tier")
).distinct() \
 .withColumn("_updated_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
 .withColumn("_updated_by",  lit(NOTEBOOK_NAME))

if table_exists(TBL_DIM_DRG):
    # SCD Type 1 — MERGE, update in place on change
    dim_drg = DeltaTable.forName(spark, TBL_DIM_DRG)
    dim_drg.alias("tgt").merge(
        df_drg_source.alias("src"),
        "tgt.drg_code = src.drg_code"
    ) \
    .whenMatchedUpdate(set={
        "drg_description":  "src.drg_description",
        "drg_severity_tier": "src.drg_severity_tier",
        "_updated_at":      "src._updated_at",
        "_updated_by":      "src._updated_by"
    }) \
    .whenNotMatchedInsertAll() \
    .execute()
    print("Type 1 MERGE complete")
else:
    # First load
    df_drg_source.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TBL_DIM_DRG)
    print("First load complete")

drg_count = spark.table(TBL_DIM_DRG).count()
print(f"\ndim_drg rows     : {drg_count:,}")
print(f"Unique DRG codes : {drg_count:,}")
display(spark.table(TBL_DIM_DRG).limit(5))

In [0]:
# ============================================================
# DIM_HOSPITAL — SCD TYPE 2
# Full history of hospital ownership and type changes
# On change: expire old row, insert new row
# ============================================================

print("=" * 55)
print("  BUILDING dim_hospital — SCD TYPE 2")
print("=" * 55)

# Tracked attributes — if any of these change, we create a new row
HOSPITAL_TRACKED_COLS = [
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "hospital_overall_rating"
]

df_hosp_source = spark.table(TBL_BRONZE_HOSPITAL_RAW).select(
    col("facility_id").alias("provider_id"),
    col("facility_name").alias("hospital_name"),
    col("hospital_type"),
    col("hospital_ownership"),
    col("emergency_services"),
    col("hospital_overall_rating"),
    col("state").alias("provider_state"),
    col("city_town").alias("provider_city"),
    col("zip_code").alias("provider_zip"),
    col("county_parish").alias("county"),
    col("telephone_number")
) \
.withColumn("_row_hash", compute_row_hash(*HOSPITAL_TRACKED_COLS)) \
.withColumn("effective_date", lit(INITIAL_DATE).cast("date")) \
.withColumn("expiry_date",    lit(END_OF_TIME).cast("date")) \
.withColumn("is_current",     lit(True)) \
.withColumn("hospital_key",
    make_surrogate_key("provider_id", "_row_hash")
) \
.withColumn("_created_at", lit(BATCH_TIMESTAMP).cast("timestamp"))

if table_exists(TBL_DIM_HOSPITAL):
    dim_hosp = DeltaTable.forName(spark, TBL_DIM_HOSPITAL)

    # Step 1 — expire changed current rows
    dim_hosp.alias("tgt").merge(
        df_hosp_source.alias("src"),
        """
        tgt.provider_id = src.provider_id
        AND tgt.is_current = true
        AND tgt._row_hash != src._row_hash
        """
    ) \
    .whenMatchedUpdate(set={
        "is_current":   "false",
        "expiry_date":  f"date_sub(current_date(), 1)"
    }) \
    .execute()
    print("Step 1 complete — expired changed rows")

    # Step 2 — insert new rows for changed + brand new providers
    existing_current = spark.table(TBL_DIM_HOSPITAL) \
        .filter("is_current = true") \
        .select("provider_id", "_row_hash")

    df_to_insert = df_hosp_source \
        .withColumn("effective_date", current_date()) \
        .join(existing_current,
              on=["provider_id", "_row_hash"],
              how="left_anti")

    if df_to_insert.count() > 0:
        df_to_insert.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(TBL_DIM_HOSPITAL)
        print(f"Step 2 complete — inserted {df_to_insert.count():,} new rows")
    else:
        print("Step 2 — no changes detected")
else:
    # First load
    df_hosp_source.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TBL_DIM_HOSPITAL)
    print("First load complete")

total   = spark.table(TBL_DIM_HOSPITAL).count()
current = spark.table(TBL_DIM_HOSPITAL).filter("is_current = true").count()
print(f"\ndim_hospital total rows    : {total:,}")
print(f"dim_hospital current rows  : {current:,}")
print(f"dim_hospital historic rows : {total - current:,}")
display(spark.table(TBL_DIM_HOSPITAL).filter("is_current = true").limit(5))

In [0]:
# ============================================================
# DIM_PAYER — SCD TYPE 2
# Tracks Medicare reimbursement rate changes per DRG
# Critical for RCM — must know which rate was active at claim time
# ============================================================

print("=" * 55)
print("  BUILDING dim_payer — SCD TYPE 2")
print("=" * 55)

PAYER_TRACKED_COLS = ["contracted_rate", "coverage_pct"]

# Derive payer rates from actual Medicare payment data
df_payer_source = spark.table(TBL_SILVER_CLAIMS) \
    .groupBy("drg_code") \
    .agg(
        expr("round(avg(avg_medicare_payment), 2)").alias("contracted_rate"),
        expr("round(avg(medicare_payment_pct), 2)").alias("coverage_pct"),
        expr("round(avg(charge_to_payment_ratio), 4)").alias("avg_ctp_ratio")
    ) \
    .withColumn("payer_id",   lit("MEDICARE_FFS")) \
    .withColumn("payer_name", lit("Medicare Fee-For-Service")) \
    .withColumn("payer_type", lit("Government")) \
    .withColumn("plan_type",  lit("Part A Inpatient")) \
    .withColumn("_row_hash",  compute_row_hash(*PAYER_TRACKED_COLS)) \
    .withColumn("effective_date", lit(INITIAL_DATE).cast("date")) \
    .withColumn("expiry_date",    lit(END_OF_TIME).cast("date")) \
    .withColumn("is_current",     lit(True)) \
    .withColumn("payer_key",
        make_surrogate_key("payer_id", "drg_code", "_row_hash")
    ) \
    .withColumn("_created_at", lit(BATCH_TIMESTAMP).cast("timestamp"))

if table_exists(TBL_DIM_PAYER):
    dim_payer = DeltaTable.forName(spark, TBL_DIM_PAYER)

    # Expire changed rows
    dim_payer.alias("tgt").merge(
        df_payer_source.alias("src"),
        """
        tgt.payer_id  = src.payer_id
        AND tgt.drg_code  = src.drg_code
        AND tgt.is_current = true
        AND tgt._row_hash != src._row_hash
        """
    ) \
    .whenMatchedUpdate(set={
        "is_current":  "false",
        "expiry_date": "date_sub(current_date(), 1)"
    }) \
    .execute()

    # Insert new rows
    existing = spark.table(TBL_DIM_PAYER) \
        .filter("is_current = true") \
        .select("payer_id", "drg_code", "_row_hash")

    df_to_insert = df_payer_source \
        .withColumn("effective_date", current_date()) \
        .join(existing,
              on=["payer_id", "drg_code", "_row_hash"],
              how="left_anti")

    if df_to_insert.count() > 0:
        df_to_insert.write \
            .format("delta").mode("append") \
            .saveAsTable(TBL_DIM_PAYER)

    print("SCD Type 2 MERGE complete")
else:
    df_payer_source.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TBL_DIM_PAYER)
    print("First load complete")

total   = spark.table(TBL_DIM_PAYER).count()
current = spark.table(TBL_DIM_PAYER).filter("is_current = true").count()
print(f"\ndim_payer total rows   : {total:,}")
print(f"dim_payer current rows : {current:,}")
display(spark.table(TBL_DIM_PAYER).filter("is_current = true").limit(5))

In [0]:
# ============================================================
# DIM_PROVIDER — SCD TYPE 2
# Tracks provider specialty, location and group practice changes
# ============================================================

print("=" * 55)
print("  BUILDING dim_provider — SCD TYPE 2")
print("=" * 55)

PROVIDER_TRACKED_COLS = [
    "provider_state",
    "hospital_type",
    "hospital_ownership",
    "rural_urban_classification"
]

df_provider_source = spark.table(TBL_SILVER_CLAIMS).select(
    col("provider_id"),
    col("provider_name"),
    col("provider_street"),
    col("provider_city"),
    col("provider_state"),
    col("provider_zip"),
    col("ruca_code"),
    col("rural_urban_classification"),
    col("hospital_type"),
    col("hospital_ownership"),
    col("emergency_services"),
    col("hospital_overall_rating")
).distinct() \
 .withColumn("_row_hash", compute_row_hash(*PROVIDER_TRACKED_COLS)) \
 .withColumn("effective_date", lit(INITIAL_DATE).cast("date")) \
 .withColumn("expiry_date",    lit(END_OF_TIME).cast("date")) \
 .withColumn("is_current",     lit(True)) \
 .withColumn("provider_key",
     make_surrogate_key("provider_id", "_row_hash")
 ) \
 .withColumn("_created_at", lit(BATCH_TIMESTAMP).cast("timestamp"))

if table_exists(TBL_DIM_PROVIDER):
    dim_prov = DeltaTable.forName(spark, TBL_DIM_PROVIDER)

    # Expire changed rows
    dim_prov.alias("tgt").merge(
        df_provider_source.alias("src"),
        """
        tgt.provider_id = src.provider_id
        AND tgt.is_current = true
        AND tgt._row_hash != src._row_hash
        """
    ) \
    .whenMatchedUpdate(set={
        "is_current":  "false",
        "expiry_date": "date_sub(current_date(), 1)"
    }) \
    .execute()

    # Insert new rows
    existing = spark.table(TBL_DIM_PROVIDER) \
        .filter("is_current = true") \
        .select("provider_id", "_row_hash")

    df_to_insert = df_provider_source \
        .withColumn("effective_date", current_date()) \
        .join(existing,
              on=["provider_id", "_row_hash"],
              how="left_anti")

    if df_to_insert.count() > 0:
        df_to_insert.write \
            .format("delta").mode("append") \
            .saveAsTable(TBL_DIM_PROVIDER)

    print("SCD Type 2 MERGE complete")
else:
    df_provider_source.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TBL_DIM_PROVIDER)
    print("First load complete")

total   = spark.table(TBL_DIM_PROVIDER).count()
current = spark.table(TBL_DIM_PROVIDER).filter("is_current = true").count()
print(f"\ndim_provider total rows   : {total:,}")
print(f"dim_provider current rows : {current:,}")
display(
    spark.table(TBL_DIM_PROVIDER)
        .filter("is_current = true")
        .select(
            "provider_key",
            "provider_id",
            "provider_name",
            "provider_state",
            "hospital_type",
            "rural_urban_classification",
            "effective_date",
            "expiry_date",
            "is_current"
        ).limit(5)
)

In [0]:
# ============================================================
# DIM_DATE — STATIC CALENDAR TABLE
# Pre-generated date dimension covering 2018-2030
# Includes CMS fiscal year periods (Oct 1 - Sep 30)
# ============================================================

print("=" * 55)
print("  BUILDING dim_date — STATIC CALENDAR")
print("=" * 55)

from pyspark.sql.functions import (
    sequence, explode, to_date, year, month,
    dayofmonth, dayofweek, quarter, date_format,
    weekofyear
)
from pyspark.sql.types import DateType

# Generate date range 2018-01-01 to 2030-12-31
df_dates = spark.sql("""
    SELECT explode(sequence(
        to_date('2018-01-01'),
        to_date('2030-12-31'),
        interval 1 day
    )) AS date_value
""") \
.withColumn("date_key",
    date_format(col("date_value"), "yyyyMMdd").cast("int")) \
.withColumn("calendar_year",     year(col("date_value"))) \
.withColumn("calendar_month",    month(col("date_value"))) \
.withColumn("calendar_quarter",  quarter(col("date_value"))) \
.withColumn("day_of_month",      dayofmonth(col("date_value"))) \
.withColumn("day_of_week",       dayofweek(col("date_value"))) \
.withColumn("week_of_year",      weekofyear(col("date_value"))) \
.withColumn("month_name",
    date_format(col("date_value"), "MMMM")) \
.withColumn("month_abbr",
    date_format(col("date_value"), "MMM")) \
.withColumn("quarter_label",
    expr("concat('Q', calendar_quarter, ' ', calendar_year)")) \
.withColumn("is_weekend",
    when(col("day_of_week").isin(1, 7), lit(True)).otherwise(lit(False))
) \
.withColumn("cms_fiscal_year",
    # CMS fiscal year runs Oct 1 - Sep 30
    when(col("calendar_month") >= 10,
        col("calendar_year") + 1
    ).otherwise(col("calendar_year"))
) \
.withColumn("cms_fiscal_quarter",
    when(col("calendar_month").isin(10, 11, 12), lit("Q1"))
    .when(col("calendar_month").isin(1, 2, 3),   lit("Q2"))
    .when(col("calendar_month").isin(4, 5, 6),   lit("Q3"))
    .otherwise(lit("Q4"))
) \
.withColumn("cms_fiscal_period",
    expr("concat(cms_fiscal_quarter, ' FY', cms_fiscal_year)")
)

df_dates.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_DIM_DATE)

date_count = spark.table(TBL_DIM_DATE).count()
print(f"\ndim_date rows : {date_count:,}")
print(f"Date range    : 2018-01-01 to 2030-12-31")

display(
    spark.table(TBL_DIM_DATE).select(
        "date_key",
        "date_value",
        "calendar_year",
        "calendar_month",
        "month_name",
        "cms_fiscal_year",
        "cms_fiscal_quarter",
        "cms_fiscal_period",
        "is_weekend"
    ).limit(10)
)

In [0]:
# ============================================================
# VERIFICATION — ALL DIMENSIONS
# ============================================================

print("=" * 55)
print("  DIMENSION VERIFICATION")
print("=" * 55)

dims = [
    (TBL_DIM_DRG,      "dim_drg",      "Type 1", None),
    (TBL_DIM_HOSPITAL, "dim_hospital", "Type 2", "is_current"),
    (TBL_DIM_PAYER,    "dim_payer",    "Type 2", "is_current"),
    (TBL_DIM_PROVIDER, "dim_provider", "Type 2", "is_current"),
    (TBL_DIM_DATE,     "dim_date",     "Static", None),
]

print(f"\n{'Dimension':<20} {'SCD':<8} {'Total':>8} {'Current':>8} {'Historic':>8}")
print("-" * 58)

for table, name, scd_type, current_col in dims:
    total = spark.table(table).count()
    if current_col:
        current  = spark.table(table).filter(f"{current_col} = true").count()
        historic = total - current
    else:
        current  = total
        historic = 0
    print(f"{name:<20} {scd_type:<8} {total:>8,} {current:>8,} {historic:>8,}")

print("\nAll dimensions verified successfully")

In [0]:
# ============================================================
# LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "silver",
    status           = "SUCCESS",
    rows_read        = spark.table(TBL_SILVER_CLAIMS).count(),
    rows_written     = (
        spark.table(TBL_DIM_DRG).count() +
        spark.table(TBL_DIM_HOSPITAL).count() +
        spark.table(TBL_DIM_PAYER).count() +
        spark.table(TBL_DIM_PROVIDER).count() +
        spark.table(TBL_DIM_DATE).count()
    ),
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"dim_drg (Type 1) | "
        f"dim_hospital (Type 2) | "
        f"dim_payer (Type 2) | "
        f"dim_provider (Type 2) | "
        f"dim_date (static 2018-2030)"
    )
)